In [ ]:
# %% Setup
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid libomp double-init abort (numpy and torch both bundle OpenMP on macOS)
os.environ.setdefault("OMP_NUM_THREADS", "1")           # avoids an OMP pthread_mutex_init segfault during inference on macOS

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import torch
from huggingface_hub import snapshot_download
from toto.data.util.dataset import MaskedTimeseries
from toto.inference.forecaster import TotoForecaster
from toto.model.toto import Toto

# Toto 1.0 (Datadog) ships as the `toto` package inside the `toto-ts` distribution. toto-ts pins exact old
# versions of everything (torch==2.7.0, numpy==1.26, ...) that don't exist for Python 3.14, so it is installed
# with `pip install --no-deps toto-ts` on top of this env (it also needs `lightning`, imported by gluonts.torch).
# Toto.from_pretrained is broken on huggingface_hub>=1.0 (it expects the removed `proxies`/`resume_download`
# kwargs), so the weights are fetched with snapshot_download and loaded through Toto's own checkpoint loader.
# Weights are Apache-2.0, no token needed.
MAX_CONTEXT = 4096   # the model doesn't truncate context itself; 4096 is the context length it was trained on

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_dir = snapshot_download("Datadog/Toto-Open-Base-1.0", allow_patterns=["config.json", "model.safetensors"])
pipeline = TotoForecaster(Toto.load_from_checkpoint(ckpt_dir, map_location=device, strict=False).model)
print(f"Loaded Toto-Open-Base-1.0 on {device}")


In [ ]:
# %% Load and align data: states (targets) + exogenous inputs (covariates)
STATE_COLUMNS = {"C_A": "C_A", "T": "T", "T_C": "T_C", "h": "h", "Q_vec": "Q", "Q_c_vec": "Qc"}
EXOG_COLUMNS = {"F1": "E_R", "F2": "U_Ac", "F4": "T_F", "F5": "C_F", "F6": "T_CF", "F7": "Q_F"}

DOWNSAMPLE = 20  # must match FACTOR in CSTR-Downsample.ipynb

with h5py.File(f"CSTR_SimulationData_ds{DOWNSAMPLE}.h5", "r") as f:
    data = {col: f[key][:, 0] for key, col in STATE_COLUMNS.items()}
    n = len(data["C_A"])

# state[j] corresponds to raw input index 1 + j, strided by DOWNSAMPLE to match the full-resolution input file
idx = 1 + np.arange(n) * DOWNSAMPLE

with h5py.File("CSTR_InputVectors.h5", "r") as f:
    for key, col in EXOG_COLUMNS.items():
        data[col] = f[key][idx, 0]
    flags = {key: f[f"{key}.plt"][idx] for key in EXOG_COLUMNS}

df = pd.DataFrame(data)
df["item_id"] = "cstr"
df["timestamp"] = pd.date_range("2026-01-01", periods=n, freq=f"{DOWNSAMPLE}s")
df = df[["item_id", "timestamp"] + list(STATE_COLUMNS.values()) + list(EXOG_COLUMNS.values())]
df.head()

In [ ]:
# %% Pick a forecast window
events = []
for key in EXOG_COLUMNS:
    active = np.where(flags[key] > 0)[0]
    if len(active) == 0:
        continue
    breaks = np.where(np.diff(active) > 1)[0]
    starts = np.concatenate([[active[0]], active[breaks + 1]])
    ends = np.concatenate([active[breaks], [active[-1]]])
    events += [(s, e, EXOG_COLUMNS[key]) for s, e in zip(starts, ends)]
events.sort()

EVENT_INDEX = 12     # which fault event to forecast across, in time order; -1 = last = most available context
BUFFER = 100          # samples of steady state before the fault kept inside the forecast window (not the context)
PLOT_LEAD_IN = 500    # samples of history shown in the plot before the forecast - display only, doesn't affect what the model sees
HORIZON = None        # forecast horizon in samples - set an int yourself, or leave None to size it to the event (capped below)

event_start, event_end, event_name = events[EVENT_INDEX]
SPLIT = max(event_start - BUFFER, 0)          # everything before this is context
context_available = SPLIT
context_used = min(context_available, MAX_CONTEXT)
if HORIZON is None:
    HORIZON = min((event_end - SPLIT) + 1000, n - SPLIT)   # Toto decodes autoregressively, so no fixed max prediction length

print(
    f"Forecasting across the {event_name} event: {context_available} samples of context available "
    f"({context_used} will actually be used - model max is {MAX_CONTEXT}), horizon = {HORIZON} steps"
)


In [ ]:
# %% Run Toto with past states + past-and-future exogenous inputs
# Toto is natively multivariate: the 6 states and the 6 exogenous inputs go in together as the 12 variates of
# one series, with the exogenous inputs LAST (Toto's convention). It decodes autoregressively one 64-step patch
# at a time, and after each patch the exogenous channels' predictions are overwritten with their known future
# values - so it only sees a change in an input from the patch after it happens, unlike TimesFM 3.0 / t0 /
# TiRex-2, which see the whole future covariate window up front. One call covers all 6 states.
# Context is sliced to context_used here since the model has no internal truncation.
# Toto is sample-based (Student-T mixture), so quantiles are taken over NUM_SAMPLES sampled paths - 256 and
# the median as point forecast, as in the Toto paper.
QUANTILES = [0.1, 0.5, 0.9]
NUM_SAMPLES = 256
SAMPLES_PER_BATCH = 64   # lower this if you run out of memory

context_df = df.iloc[SPLIT - context_used:SPLIT].copy()
future_df = df.iloc[SPLIT:SPLIT + HORIZON][["item_id", "timestamp"] + list(EXOG_COLUMNS.values())].copy()
truth_df = df.iloc[SPLIT:SPLIT + HORIZON].copy()

series = torch.tensor(                                                                   # (12, context_used): states then exog
    context_df[list(STATE_COLUMNS.values()) + list(EXOG_COLUMNS.values())].to_numpy(dtype=np.float32).T, device=device
)
future_exog = torch.tensor(future_df[list(EXOG_COLUMNS.values())].to_numpy(dtype=np.float32).T, device=device)  # (6, HORIZON)
timestamps = torch.tensor(context_df["timestamp"].astype("int64").to_numpy() // 10**9, device=device)

inputs = MaskedTimeseries(
    series=series,
    padding_mask=torch.ones_like(series, dtype=torch.bool),
    id_mask=torch.zeros_like(series),
    timestamp_seconds=timestamps.expand_as(series),
    time_interval_seconds=torch.full((series.shape[0],), DOWNSAMPLE, device=device),
    num_exogenous_variables=len(EXOG_COLUMNS),
)
out = pipeline.forecast(
    inputs,
    prediction_length=HORIZON,
    num_samples=NUM_SAMPLES,
    samples_per_batch=SAMPLES_PER_BATCH,
    future_exogenous_variables=future_exog,
)   # out.samples: (1, 12, HORIZON, NUM_SAMPLES) - the first 6 variates are the states


quantiles = {q: out.quantile(q)[0].cpu().numpy() for q in QUANTILES}   # each (12, HORIZON)
pred_df = pd.concat(
    [
        pd.DataFrame({
            "item_id": "cstr",
            "timestamp": truth_df["timestamp"].values,
            "target_name": col,
            "predictions": quantiles[0.5][v],
            **{str(q): quantiles[q][v] for q in QUANTILES},
        })
        for v, col in enumerate(STATE_COLUMNS.values())
    ],
    ignore_index=True,
)


pred_df.head()


In [ ]:
# %% Accuracy vs. the held-out actual simulation, and plots per state
for col in STATE_COLUMNS.values():
    sub = pred_df[pred_df["target_name"] == col].reset_index(drop=True)
    mae = np.mean(np.abs(sub["predictions"].values - truth_df[col].values))
    print(f"{col}: MAE = {mae:.5g}")

context_tail = context_df.iloc[-PLOT_LEAD_IN:]
fig, axes = plt.subplots(3, 2, figsize=(12, 10))
for ax, col in zip(axes.ravel(), STATE_COLUMNS.values()):
    sub = pred_df[pred_df["target_name"] == col]
    ax.plot(context_tail["timestamp"], context_tail[col], color="black", label="history")
    ax.plot(truth_df["timestamp"], truth_df[col], color="black", linestyle="--", label="actual (held out)")
    ax.plot(sub["timestamp"], sub["predictions"], color="tab:blue", label="Toto forecast")
    ax.fill_between(sub["timestamp"], sub["0.1"], sub["0.9"], color="tab:blue", alpha=0.2, label="10-90% interval")
    ax.axvline(truth_df["timestamp"].iloc[0], color="gray", linestyle=":")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=30)
axes.ravel()[0].legend(fontsize=8)
fig.tight_layout()